In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_context("poster")
sns.set_style("ticks")

In [ ]:
fi = pd.read_parquet("0.parquet")
spearman = pd.read_parquet("1.parquet")
weights = pd.read_parquet("2.parquet")
weights["AbsWeight"] = weights.Weight.abs()

In [ ]:
features_fi = (
    fi[(fi.split == "test") & (fi["trainer.representations.noise_level"] == 0)]
    .groupby(["Feature", "trainer.model_builder.param"])["mean"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .groupby("trainer.model_builder.param")
    .head(5)[["Feature", "trainer.model_builder.param"]]
)

In [ ]:
features_weights = (
    weights[weights["trainer.representations.noise_level"] == 0]
    .groupby(["Feature", "trainer.model_builder.param"])
    .AbsWeight.mean()
    .sort_values(ascending=False)
    .reset_index()
    .groupby("trainer.model_builder.param")
    .head(5)[["Feature", "trainer.model_builder.param"]]
)

In [ ]:
features = pd.concat([features_fi, features_weights])[["Feature"]].drop_duplicates()

# FI

In [ ]:
g = sns.relplot(
    fi[fi.split == "test"].merge(features),
    kind="line",
    x="trainer.representations.noise_level",
    y="mean",
    hue="Feature",
    hue_order=features.Feature,
    col="trainer.model_builder.param",
    aspect=1.5,
    errorbar="sd",
    marker="o",
    markersize=6,
)
g.set_ylabels("FI")
g.set_xlabels("Artificial Noise Level")
g.set_titles("{col_name}")
g.refline(y=0, linestyle="--", color="black", linewidth=1)
plt.savefig(
    "../../../paper/figs/BERT_RC_CLS/artificial_noise/feature_importance.pdf",
    bbox_inches="tight",
)
plt.show()

# Weights

In [ ]:
g = sns.relplot(
    weights.merge(features),
    kind="line",
    x="trainer.representations.noise_level",
    y="Weight",
    hue="Feature",
    hue_order=features.Feature,
    col="trainer.model_builder.param",
    aspect=1.5,
    errorbar="sd",
    marker="o",
    markersize=6,
)
g.set_xlabels("Artificial Noise Level")
g.set_titles("{col_name}")
g.refline(y=0, linestyle="--", color="black", linewidth=1)
plt.savefig(
    "../../../paper/figs/BERT_RC_CLS/artificial_noise/weights.pdf", bbox_inches="tight"
)
plt.show()

# Training duration

In [ ]:
ax = sns.lineplot(
    weights,
    x="trainer.representations.noise_level",
    y="training_duration",
    hue="trainer.model_builder.param",
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.set_ylabel("Training Duration (s)")
ax.set_xlabel("Artificial Noise Level")
ax.get_legend().set_title(None)
plt.savefig(
    "../../../paper/figs/BERT_RC_CLS/artificial_noise/training_duration.pdf",
    bbox_inches="tight",
)
plt.show()

# Spearman

In [ ]:
ax = sns.lineplot(
    spearman[spearman.split == "test"],
    x="trainer.representations.noise_level",
    y="mean",
    hue="trainer.model_builder.param",
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.set_ylabel("Spearman")
ax.set_xlabel("Artificial Noise level")
ax.get_legend().set_title(None)
plt.savefig(
    "../../../paper/figs/BERT_RC_CLS/artificial_noise/spearman.pdf", bbox_inches="tight"
)
plt.show()